# Fashion Embeddings with ZooClaw-FashionSigLIP2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SerendipityOneInc/ZooData-Notebook/blob/main/e-commerce-search/fashion_embeddings.ipynb)

Get raw **768-dim fashion embeddings** straight from our fine-tuned model — no GPU, no model download, no infra. This notebook is a focused tour of the three low-level model endpoints you'd use to build a **custom visual-search index**, a **reranker**, or a **zero-shot classifier**.

### Powered by ZooClaw-FashionSigLIP2

[`srpone/zooclaw-fashionsiglip2`](https://huggingface.co/srpone/zooclaw-fashionsiglip2) is a SigLIP2 ViT-B/16 encoder (~375M params) distilled and fine-tuned on a large-scale fashion-retrieval corpus. Both the image and text towers project into a **shared 768-dim, L2-normalized** space — so cosine similarity is just a dot product. On the public [LookBench](https://github.com/SerendipityOneInc/look-bench) fashion-retrieval benchmark it is currently the best open-source fashion vision-language model. See the [paper](https://arxiv.org/abs/2606.27708) for the full recipe.

| Endpoint | What it returns | Batch limit | Cost |
|----------|-----------------|-------------|------|
| `POST /model/fashion-image-embedding` | 768-dim vector per image | 8 images | 1 credit |
| `POST /model/fashion-text-embedding` | 768-dim vector per query | 32 queries | 1 credit |
| `POST /model/fashion-similarity` | `[nText x nImage]` cosine matrix | 32 x 8 | 1 credit |

## Setup

Sign up at [zoodata.ai/register](https://zoodata.ai/register) to get your API key — every new account includes 1,000 free credits to get started. Need more? See the [credit packages](https://zoodata.ai/pricing) for higher volumes.

In [ ]:
!pip install -q httpx numpy Pillow matplotlib

In [ ]:
import getpass
import os

if "ZOODATA_API_KEY" not in os.environ:
    os.environ["ZOODATA_API_KEY"] = getpass.getpass("Enter your ZooData API key (hms_live_...): ")

API_KEY = os.environ["ZOODATA_API_KEY"]
BASE_URL = "https://api.zoodata.ai/openapi/v2"

In [ ]:
import httpx
import numpy as np
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt

client = httpx.Client(
    base_url=BASE_URL,
    headers={"Authorization": f"Bearer {API_KEY}"},
    timeout=30.0,
)


# ── SigLIP2 model wrappers ───────────────────────────────────────────────
def embed_images(image_urls: list[str]) -> np.ndarray:
    """POST /model/fashion-image-embedding -> [N, 768] float32. Max 8 HTTPS URLs."""
    resp = client.post("/model/fashion-image-embedding", json={"imageUrls": image_urls})
    resp.raise_for_status()
    items = sorted(resp.json()["data"]["embeddings"], key=lambda e: e["imageIndex"])
    return np.array([e["embeddingVector"] for e in items], dtype=np.float32)


def embed_texts(queries: list[str]) -> np.ndarray:
    """POST /model/fashion-text-embedding -> [M, 768] float32. Max 32 queries."""
    resp = client.post("/model/fashion-text-embedding", json={"queries": queries})
    resp.raise_for_status()
    items = resp.json()["data"]["embeddings"]  # already in input order
    return np.array([e["embeddingVector"] for e in items], dtype=np.float32)


def text_image_similarity(text_queries: list[str], image_urls: list[str]) -> np.ndarray:
    """POST /model/fashion-similarity -> [nText, nImage] cosine matrix in one call."""
    resp = client.post(
        "/model/fashion-similarity",
        json={"textQueries": text_queries, "imageUrls": image_urls},
    )
    resp.raise_for_status()
    return np.array(resp.json()["data"]["similarityScores"], dtype=np.float32)


# ── Helpers (demo images come from the product-search endpoint) ───────────
def get_demo_images(query: str, k: int = 6) -> list[str]:
    """Grab a few clean product image URLs to use as demo inputs."""
    resp = client.post("/model/fashion-product-search", json={"query": query, "topK": k})
    resp.raise_for_status()
    products = resp.json()["data"]["products"]
    return [p["imageUrl"] for p in products if p.get("imageUrl")][:k]


def load_image(url: str) -> Image.Image:
    r = httpx.get(url, timeout=10, follow_redirects=True)
    return Image.open(BytesIO(r.content))

---
## Task 1: Image Embeddings

Encode product images into 768-dim vectors with `POST /model/fashion-image-embedding` (batch up to 8). The vectors are L2-normalized, so every row has unit length.

In [ ]:
# Grab a small demo catalog of handbag images
catalog_urls = get_demo_images("women handbag", k=6)
print(f"Got {len(catalog_urls)} demo images")

img_embs = embed_images(catalog_urls)
print(f"Image embedding matrix: {img_embs.shape}  (dtype={img_embs.dtype})")
print(f"L2 norm of first vector: {np.linalg.norm(img_embs[0]):.4f}  (expect ~1.0)")
print(f"First 5 dims of vec[0]: {img_embs[0, :5]}")

---
## Task 2: Text Embeddings + Client-Side Ranking

Encode text with `POST /model/fashion-text-embedding`, then rank the images from Task 1 against each query. Because both towers share the same normalized space, **cosine similarity is just a dot product**.

In [ ]:
queries = [
    "red leather crossbody bag",
    "black quilted chain shoulder bag",
    "beige straw beach tote",
]
txt_embs = embed_texts(queries)
print(f"Text embedding matrix: {txt_embs.shape}\n")

# Cosine similarity == dot product (both sides are L2-normalized)
scores = txt_embs @ img_embs.T          # shape [n_queries, n_images]

for i, q in enumerate(queries):
    top = np.argsort(-scores[i])[:3]
    print(f'Query: "{q}"')
    for rank, j in enumerate(top, 1):
        print(f"  #{rank}  score={scores[i, j]:.3f}  {catalog_urls[j][:70]}...")
    print()

In [ ]:
# Visualize: each row is a query, showing its top-3 image matches
fig, axes = plt.subplots(len(queries), 4, figsize=(14, 3.4 * len(queries)))
for i, q in enumerate(queries):
    top = np.argsort(-scores[i])[:3]
    axes[i, 0].text(0.5, 0.5, q, ha="center", va="center", wrap=True, fontsize=11, fontweight="bold")
    axes[i, 0].axis("off")
    for c, j in enumerate(top, 1):
        try:
            axes[i, c].imshow(load_image(catalog_urls[j]))
        except Exception:
            axes[i, c].text(0.5, 0.5, "N/A", ha="center", va="center")
        axes[i, c].set_title(f"score {scores[i, j]:.3f}", fontsize=9)
        axes[i, c].axis("off")
plt.tight_layout()
plt.show()

---
## Task 3: One-Shot Similarity Matrix

When you just want the score matrix, `POST /model/fashion-similarity` embeds both sides server-side and returns the `[nText x nImage]` cosine matrix in a single call — no need to manage embeddings yourself.

In [ ]:
matrix = text_image_similarity(queries, catalog_urls)
print(f"similarityScores shape: {matrix.shape}  (nText x nImage)\n")

# Pretty-print
header = "query \\ image   " + "  ".join(f"img[{j}]" for j in range(len(catalog_urls)))
print(header)
for i, q in enumerate(queries):
    row = "  ".join(f"{matrix[i, j]:.3f}" for j in range(matrix.shape[1]))
    print(f"  q[{i}]         {row}")
print("\nLegend:")
for i, q in enumerate(queries):
    print(f"  q[{i}] = {q}")

# Sanity check: server matrix should match the client-side dot product from Task 2
print(f"\nMax |client_dot - server_matrix| = {np.max(np.abs(scores - matrix)):.2e}  (expect ~0)")

---
## Task 4: Zero-Shot Fashion Classification

No training needed — describe your candidate labels as text, then pick the highest-scoring one per image. This turns the encoder into an open-vocabulary classifier for attributes, categories, styles, or materials.

In [ ]:
label_prompts = [
    "a handbag", "a pair of shoes", "a dress",
    "a watch", "sunglasses", "a hat",
]

# Score every demo image against every candidate label in one call
cls_matrix = text_image_similarity(label_prompts, catalog_urls)   # [nLabels, nImages]

for j, url in enumerate(catalog_urls):
    col = cls_matrix[:, j]
    best = int(np.argmax(col))
    print(f"img[{j}] -> {label_prompts[best]:<16} (score {col[best]:.3f})  {url[:55]}...")

---
## Task 5: Build a Mini Text-to-Image Search Index

The production pattern: **embed your catalog once, cache the vectors, then serve text queries with a cheap dot product**. Here we encode a small gallery, then retrieve the best matches for an arbitrary text query entirely client-side.

In [ ]:
# 1. Build a gallery (embed once, then reuse)
gallery_urls = get_demo_images("women summer dress", k=8)
gallery_embs = embed_images(gallery_urls)   # cache this [N, 768] in your vector DB
print(f"Indexed {gallery_embs.shape[0]} items.")

# 2. Serve a text query — only needs one text-embedding call + a dot product
user_query = "floral off-shoulder maxi dress"
q_emb = embed_texts([user_query])[0]        # [768]
ranking = np.argsort(-(gallery_embs @ q_emb))[:4]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, j in zip(axes, ranking):
    try:
        ax.imshow(load_image(gallery_urls[j]))
    except Exception:
        ax.text(0.5, 0.5, "N/A", ha="center", va="center")
    ax.set_title(f"score {gallery_embs[j] @ q_emb:.3f}", fontsize=10)
    ax.axis("off")
fig.suptitle(f'Query: "{user_query}"', fontsize=13)
plt.tight_layout()
plt.show()

---
## Summary

You now have direct access to the `ZooClaw-FashionSigLIP2` model through three endpoints — enough to build custom search, ranking, and classification on top of a fashion-tuned embedding space.

### Endpoints used

| Endpoint | Purpose | Cost |
|----------|---------|------|
| `POST /model/fashion-image-embedding` | 768-dim image vectors (batch up to 8) | 1 credit |
| `POST /model/fashion-text-embedding` | 768-dim text vectors (batch up to 32) | 1 credit |
| `POST /model/fashion-similarity` | one-shot `[nText x nImage]` cosine matrix | 1 credit |

### Key facts

- **Shared space** — image and text vectors are directly comparable; cosine similarity = dot product (vectors are L2-normalized by default; pass `normalizeVectors: false` for raw vectors).
- **768 dimensions** — compact enough to index millions of products cheaply.
- **Embed once, query forever** — cache image embeddings in your vector DB; only new queries cost a call.

### Next steps

- **[Fashion Image Search](https://colab.research.google.com/github/SerendipityOneInc/ZooData-Notebook/blob/main/e-commerce-search/image_search.ipynb)** — end-to-end visual search + detect → embed → search "Shop the Look" pipeline.
- **[Fashion Product Search](https://colab.research.google.com/github/SerendipityOneInc/ZooData-Notebook/blob/main/e-commerce-search/product_search.ipynb)** — natural-language product discovery with brand / price / retailer filters.
- Load the open weights directly from [srpone/zooclaw-fashionsiglip2](https://huggingface.co/srpone/zooclaw-fashionsiglip2) if you'd rather self-host.